In [1]:
import os
print(os.getcwd())   # xem kernel đang đứng ở đâu


d:\ct551_v2


In [ ]:
import re
from datetime import datetime
import numpy as np
import pandas as pd

path_dataset = "dataset_high/HI-Small_Patterns.txt"   

def parse_pattern(path):
    pattern_list = []
    pattern_type = None
    pattern_ts = []
    with open(path) as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                continue
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):      
                rest = line.split("-", 1)[1].strip()
                pattern_type = rest.split(":")[0].strip()
                pattern_ts = []
            elif line.startswith("END LAUNDERING ATTEMPT"):
                pattern_list.append((pattern_type, pattern_ts))
                pattern_type = None
                pattern_ts = []
            else:
                ts_str = line.split(",", 1)[0]
                pattern_ts.append(datetime.strptime(ts_str, "%Y/%m/%d %H:%M")) 
        # fix: return đặt sau khi vòng for chạy hết, không nằm trong nhánh else
    return pattern_list

def summarize(pattern_list):
    rows = []
    for pattern_type, ts_list in pattern_list:
        n_tx = len(ts_list)
        span_hour = (max(ts_list) - min(ts_list)).total_seconds() / 3600
        rows.append({"pattern": pattern_type, "n_tx": n_tx, "span_h": span_hour})
    df = pd.DataFrame(rows)
    summary = df.groupby("pattern").agg(
        n_transaction=("n_tx", "size"),
        median_tx=("n_tx", "median"), #trung vị của số lần giao dịch
        median_span=("span_h", "median"),   #trung bình thời gian                       
        p90_span=("span_h", lambda s: np.percentile(s, 90)), #thời gian 1 phi vụ thực hiện ở 0.1n lần 
    ).round(2)
    return summary.sort_values("median_span")

if __name__ == "__main__":
    pattern_list = parse_pattern(path_dataset)          
    summary = summarize(pattern_list)
    print(summary)
    print(f"total_attempts: {len(pattern_list)}")       


                n_transaction  median_tx  median_span  p90_span
pattern                                                        
BIPARTITE                  49        4.0        24.33     43.12
RANDOM                     41        3.0        46.00     86.63
CYCLE                      54        4.0        71.88     90.35
STACK                      43       10.0        73.32    101.00
FAN-OUT                    48        7.0        76.66     94.71
FAN-IN                     40        8.0        84.93     94.70
SCATTER-GATHER             44       14.0        88.77     95.50
GATHER-SCATTER             51       14.0       150.82    183.83
total_attempts: 370


In [1]:
# ==== Độ quan trọng của is_round trong các pattern chuyển tiền ====
import numpy as np, pandas as pd

PATH_PAT = "dataset_high/HI-Small_Patterns.txt"
PATH_TRX = "dataset_high/HI-Small_Trans.csv"
COLS = ["Timestamp","From Bank","Account","To Bank","Account.1","Amount Received",
        "Receiving Currency","Amount Paid","Payment Currency","Payment Format","Is Laundering"]

def parse_pattern_rows(path):
    """Giống parse_pattern nhưng giữ TOÀN BỘ cột giao dịch, không chỉ timestamp."""
    rows, ptype, aid = [], None, -1
    with open(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith("BEGIN LAUNDERING ATTEMPT"):
                ptype = line.split("-", 1)[1].split(":")[0].strip()
                aid += 1
            elif line.startswith("END LAUNDERING ATTEMPT"):
                ptype = None
            else:
                rows.append([ptype, aid] + line.split(","))
    df = pd.DataFrame(rows, columns=["pattern", "attempt_id"] + COLS)
    df["Amount Paid"] = df["Amount Paid"].astype(float)
    return df

def add_round_flags(df):
    """round_1000 = đúng định nghĩa is_round trong feature_node.py; thêm mốc 100 để đối chiếu."""
    a = df["Amount Paid"].to_numpy()
    df["round_1000"] = ((a % 1000 == 0) & (a > 0)).astype("int8")
    df["round_100"]  = ((a %  100 == 0) & (a > 0)).astype("int8")
    return df

pat = add_round_flags(parse_pattern_rows(PATH_PAT))
trx = add_round_flags(pd.read_csv(PATH_TRX,
        usecols=["Amount Paid", "Payment Currency", "Is Laundering"]))
bg  = trx[trx["Is Laundering"] == 0]          # nhóm nền: giao dịch sạch

print(f"pattern: {len(pat):,} tx / {pat['attempt_id'].nunique()} attempt | "
      f"nền (Is Laundering==0): {len(bg):,} tx")

def report(flag):
    base = bg[flag].mean()
    rows = []
    for p, g in list(pat.groupby("pattern")) + [("== TẤT CẢ PATTERN ==", pat)]:
        k, n = int(g[flag].sum()), len(g)
        rows.append({"pattern": p, "n_tx": n, "n_round": k,
                     "rate_%": 100*k/n, "base_%": 100*base,
                     "lift": (k/n)/base if base else np.nan,
                     "exp_round": n*base})      # số tx tròn KỲ VỌNG nếu pattern giống nền
    return pd.DataFrame(rows).sort_values("lift", ascending=False), base

for flag in ["round_1000", "round_100"]:
    tb, base = report(flag)
    print(f"\n### {flag}  (nền = {100*base:.4f}%)")
    print(tb.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# đối chứng 1: toàn bộ nhãn Is Laundering==1 trong CSV (nhiều hơn Patterns.txt)
lau = trx[trx["Is Laundering"] == 1]
print(f"\nĐối chứng | Is Laundering==1: {len(lau):,} tx -> "
      f"round_1000={int(lau['round_1000'].sum())}, round_100={int(lau['round_100'].sum())}")

# đối chứng 2: is_round có phải chỉ là dấu hiệu của loại tiền tệ không?
print("\nTỷ lệ round_1000 của nhóm nền theo Payment Currency (%):")
print((100*bg.groupby("Payment Currency")["round_1000"].mean())
      .sort_values(ascending=False).round(4).to_string())

pattern: 3,209 tx / 370 attempt | nền (Is Laundering==0): 5,073,168 tx

### round_1000  (nền = 0.0013%)
             pattern  n_tx  n_round  rate_%  base_%   lift  exp_round
           BIPARTITE   263        0  0.0000  0.0013 0.0000     0.0035
               CYCLE   287        0  0.0000  0.0013 0.0000     0.0038
              FAN-IN   318        0  0.0000  0.0013 0.0000     0.0043
             FAN-OUT   342        0  0.0000  0.0013 0.0000     0.0046
      GATHER-SCATTER   716        0  0.0000  0.0013 0.0000     0.0096
              RANDOM   191        0  0.0000  0.0013 0.0000     0.0026
      SCATTER-GATHER   626        0  0.0000  0.0013 0.0000     0.0084
               STACK   466        0  0.0000  0.0013 0.0000     0.0062
== TẤT CẢ PATTERN ==  3209        0  0.0000  0.0013 0.0000     0.0430

### round_100  (nền = 0.0123%)
             pattern  n_tx  n_round  rate_%  base_%   lift  exp_round
           BIPARTITE   263        0  0.0000  0.0123 0.0000     0.0325
               CYCLE   2